In [1]:
import pandas as pd
import numpy as np

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os

def preparar_fma_final(tracks_path, features_path, genres_path, min_instances=20):
    print("1/5 - Carregando metadados e features...")
    # Carregamento com multi-index do FMA
    tracks = pd.read_csv(tracks_path, index_col=0, header=[0, 1])
    features = pd.read_csv(features_path, index_col=0, header=[0, 1, 2])
    genres = pd.read_csv(genres_path, index_col=0)
    
    # 2/5 - Reconstruindo a Árvore Hierárquica
    print("2/5 - Expandindo a hierarquia de gêneros...")
    genre_map = genres[['title', 'parent']].to_dict('index')

    def get_full_path(genre_id):
        if pd.isna(genre_id) or genre_id not in genre_map:
            return "Root"
        name = str(genre_map[genre_id]['title']).replace(".", "_").replace("/", "-")
        parent_id = genre_map[genre_id]['parent']
        if parent_id == 0 or pd.isna(parent_id):
            return f"Root.{name}"
        return f"{get_full_path(parent_id)}.{name}"

    def extract_deepest_label(genres_list_str):
        try:
            g_list = eval(genres_list_str)
            if not g_list: return None
            paths = [get_full_path(gid) for gid in g_list]
            # Retorna o caminho mais longo (mais específico)
            return max(paths, key=lambda p: len(p.split('.')))
        except:
            return None

    # Aplicando mapeamento
    labels_series = tracks[('track', 'genres_all')].apply(extract_deepest_label)
    
    # 3/5 - Alinhando Features e Labels
    print("3/5 - Alinhando dados e limpando classes raras...")
    # Filtramos apenas tracks que possuem features e labels válidas
    common_index = features.index.intersection(labels_series.dropna().index)
    X_raw = features.loc[common_index].values # Matriz 2D (N, 518)
    y_raw = labels_series.loc[common_index].values # Array de strings
    
    # Filtro de instâncias mínimas para permitir split estratificado
    counts = pd.Series(y_raw).value_counts()
    valid_classes = counts[counts >= min_instances].index
    mask = np.isin(y_raw, valid_classes)
    
    X_filtered = X_raw[mask]
    y_filtered = y_raw[mask]

    print(f"Dataset final: {len(y_filtered)} instâncias em {len(valid_classes)} classes.")

    # 4/5 - Split Estratificado Seguro
    print("4/5 - Executando split estratificado...")
    from sklearn.preprocessing import LabelEncoder

    le = LabelEncoder()
    y_encoded = le.fit_transform(y_filtered)

    # X_filtered já é uma matriz 2D NumPy, o que evita o erro de "scalar index"
    X_train, X_test, y_train_idx, y_test_idx = train_test_split(
        X_filtered, y_encoded, 
        test_size=0.3, 
        random_state=42, 
        stratify=y_filtered
    )

    y_train = le.inverse_transform(y_train_idx)
    y_test = le.inverse_transform(y_test_idx)

    # 5/5 - Salvando em Parquet
    print("5/5 - Salvando arquivos Parquet...")
    
    def save_to_parquet(X, y, filename):
        # Para o Parquet aceitar o embedding, convertemos cada linha da matriz em uma lista
        df = pd.DataFrame({
            'embeddings': list(X),
            'label': y
        })
        df.to_parquet(filename, index=False, engine='pyarrow')
        print(f"Salvo: {filename} | Shape: {X.shape}")

    save_to_parquet(X_train, y_train, "../data/multi-class/fma/train.parquet")
    save_to_parquet(X_test, y_test, "../data/multi-class/fma/test.parquet")
    
    # Metadados rápidos
    max_depth = max([len(l.split('.')) for l in y_filtered])
    print(f"\nConcluído!")
    print(f"Profundidade máxima da hierarquia: {max_depth}")
    print(f"Classes no Nível 1: {len(set([l.split('.')[1] for l in y_filtered]))}")

# Execução (Certifique-se de que os arquivos estão na pasta)
preparar_fma_final('../data/multi-class/fma/fma_metadata/tracks.csv', '../data/multi-class/fma/fma_metadata/features.csv', '../data/multi-class/fma/fma_metadata/genres.csv')

1/5 - Carregando metadados e features...
2/5 - Expandindo a hierarquia de gêneros...
3/5 - Alinhando dados e limpando classes raras...
Dataset final: 104248 instâncias em 148 classes.
4/5 - Executando split estratificado...
5/5 - Salvando arquivos Parquet...
Salvo: ../data/multi-class/fma/train.parquet | Shape: (72973, 518)
Salvo: ../data/multi-class/fma/test.parquet | Shape: (31275, 518)

Concluído!
Profundidade máxima da hierarquia: 6
Classes no Nível 1: 16


In [14]:
pd.read_parquet('../data/multi-class/fma/train.parquet', engine='pyarrow').head()

,embeddings,label
0,"[0.54776012897, 0.052311141044, -0.28284466267...",Root.Electronic
1,"[-0.28921937943, -0.37005597353, 0.35129776597...",Root.Electronic
2,"[-0.51655083895, -0.27197483182, 1.1218944788,...",Root.Experimental.Noise
3,"[0.75935006142, -0.78383225203, -0.77386540174...",Root.Rock.Loud-Rock.Noise-Rock
4,"[-0.57452559471, -0.70213741064, 0.18841914833...",Root.Rock.Industrial


In [2]:
fma = pd.read_csv('../data/multi-class/fma/fma_hierarchical.csv')

fma.columns

Index(['track_id', 'chroma_cens_kurtosis_01', 'chroma_cens_kurtosis_02',
       'chroma_cens_kurtosis_03', 'chroma_cens_kurtosis_04',
       'chroma_cens_kurtosis_05', 'chroma_cens_kurtosis_06',
       'chroma_cens_kurtosis_07', 'chroma_cens_kurtosis_08',
       'chroma_cens_kurtosis_09',
       ...
       'tonnetz_std_05', 'tonnetz_std_06', 'zcr_kurtosis_01', 'zcr_max_01',
       'zcr_mean_01', 'zcr_median_01', 'zcr_min_01', 'zcr_skew_01',
       'zcr_std_01', 'label'],
      dtype='str', length=520)

In [4]:
def compact_fma_to_embeddings(input_csv, output_csv):
    df = pd.read_csv(input_csv, index_col=0)
    
    # Identifica as colunas de features (todas exceto a label)
    feature_cols = [c for c in df.columns if c != 'label']
    
    print(f"Compactando {len(feature_cols)} colunas em um vetor de embeddings...")
    
    # Criamos a coluna 'embeddings' convertendo as features em listas
    # Usamos .values para velocidade
    df['embeddings'] = df[feature_cols].values.tolist()
    
    # Mantemos apenas o essencial
    df_compact = df[['embeddings', 'label']]
    
    # Salva o CSV. Nota: a lista será salva como uma string no CSV.
    df_compact.to_csv(output_csv)
    print(f"Sucesso! Dataset compacto salvo em: {output_csv}")

compact_fma_to_embeddings('../data/multi-class/fma/fma_hierarchical.csv', '../data/multi-class/fma/fma_embeddings.csv')

Compactando 518 colunas em um vetor de embeddings...


/tmp/ipykernel_3642356/187629531.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['embeddings'] = df[feature_cols].values.tolist()


Sucesso! Dataset compacto salvo em: ../data/multi-class/fma/fma_embeddings.csv


In [7]:
import pandas as pd
import numpy as np
import ast
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/multi-class/fma/fma_embeddings.csv")

X = np.vstack(df['embeddings'].values)
y = df['label'].values

X.shape

(41598, 1)

In [9]:
y

<ArrowStringArray>
[     'Root.Hip-Hop',      'Root.Hip-Hop',         'Root.Rock',
         'Root.Rock', 'Root.Experimental', 'Root.Experimental',
         'Root.Folk',         'Root.Folk',         'Root.Jazz',
         'Root.Jazz',
 ...
 'Root.Experimental', 'Root.Experimental', 'Root.Experimental',
 'Root.Experimental',         'Root.Rock',         'Root.Rock',
         'Root.Rock',         'Root.Rock',         'Root.Rock',
         'Root.Rock']
Length: 41598, dtype: str

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

TypeError: only integer scalar arrays can be converted to a scalar index

In [ ]:
def salvar_datasets_parquet(X_train, y_train, X_test, y_test, prefixo="fma"):
    # 1. Converter matrizes para DataFrames
    # Transformamos o array X em uma lista de arrays para que o Pandas trate cada embedding como uma célula
    df_train = pd.DataFrame({
        'embeddings': list(X_train),
        'label': y_train
    })
    
    df_test = pd.DataFrame({
        'embeddings': list(X_test),
        'label': y_test
    })
    
    # 2. Salvar usando o motor 'pyarrow' ou 'fastparquet'
    # O formato parquet suporta nativamente tipos complexos (listas)
    train_file = f"../data/multi-class/fma/train.parquet"
    test_file = f"../data/multi-class/fma/test.parquet"
    
    df_train.to_parquet(train_file, index=False, engine='pyarrow')
    df_test.to_parquet(test_file, index=False, engine='pyarrow')
    
    print(f"Dataset de Treino salvo: {train_file} ({len(df_train)} instâncias)")
    print(f"Dataset de Teste salvo: {test_file} ({len(df_test)} instâncias)")

# Execução
salvar_datasets_parquet(X_train, y_train, X_test, y_test)

ArrowKeyError: A type extension with name pandas.period already defined

In [10]:
!pip install pyarrow